# Strict splits and matched-negative candidate design

**Status:** drafting code for review.

**When to run:** after notebook 00 identifies overlap or shortcut risk.

**Inputs:** shared split CSVs. Cold-family, cross-species, temporal, and
matched-negative paths additionally require reviewed metadata files.

**Outputs:** split manifests and candidate review tables only when the
corresponding write switch is enabled.

**Safety:** unknown pairs are treated as *candidates*, never silently as
confirmed negatives. The notebook writes to `splits_ablations/`, never to
the original `splits/` directory.

**Working label definition:** `on-target` means a pair in SNOOPPI's
documented positive split; `off-target` means a pair in its documented
negative split. The unknown split is neither class. Replace these terms
with assay-specific definitions if the project has a narrower target/off-
target ontology, and retain the evidence source for every negative.

**Shared protocol:** set `SNOOPPI_ROOT` and `SNOOPPI_SPLIT_DIR` before setup.
Use the same split directory in every draft. Existing caches use original raw
sequence keys; this suite does not regenerate embeddings. Metadata uses full
sequence hashes, while leakage audits also check normalized 1,024-residue inputs.


## 1. Setup and protocol controls

In [ ]:
from pathlib import Path
import os
import sys

# Code location and data/artifact location are separate. Mount Drive first in Colab:
# from google.colab import drive
# drive.mount("/content/drive")
# Set these environment variables BEFORE running this cell, consistently in every notebook:
# os.environ["SNOOPPI_ROOT"] = "/content/drive/MyDrive/SNOOPPI"
# os.environ["SNOOPPI_SPLIT_DIR"] = "splits"  # or "splits_ablations/cold_cluster_seed44_v2"
# os.environ["SNOOPPI_REPO_ROOT"] = "/content/SNOOPPI"
repository_root = Path(os.environ.get("SNOOPPI_REPO_ROOT", "/content/SNOOPPI"))
candidates = [Path.cwd(), Path.cwd() / "drafting_code", Path.cwd().parent / "drafting_code",
              repository_root / "drafting_code"]
DRAFT_DIR = next((p for p in candidates if (p / "snooppi_utils.py").is_file()), None)
if DRAFT_DIR is None:
    raise FileNotFoundError("Cannot locate drafting_code/snooppi_utils.py; set SNOOPPI_REPO_ROOT to the clone root")
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

from snooppi_utils import (
    LABEL, SEQUENCE_A, SEQUENCE_B, canonical_pair_id, full_pair_id,
    experiment_paths, load_splits, normalize_sequence, sequence_id,
    full_sequence_id, split_manifest, cache_paths, add_pair_ids,
    assert_model_ready, validate_cache_coverage, preprocessing_report,
    require_comparable_preprocessing, file_sha256, make_run_manifest,
    prediction_identity, validate_prediction_identity, align_paired_predictions,
    prepare_outputs, save_json,
)

PATHS = experiment_paths()

# ===== EDIT YOUR GOOGLE DRIVE FILE PATHS HERE =====
# The uploaded train/validation/test CSVs contain sequences and labels.
# Replace any value below with Path("/content/drive/MyDrive/YOUR_FOLDER/your_file.csv").
# Copy the SAME three CSV paths into every draft notebook. Existing CSVs are read directly;
# this setup does not generate splits or change their contents.
SPLIT_CSV_FILES = {
    "train": PATHS["splits"] / "train.csv",            # TODO: your cached training CSV
    "validation": PATHS["splits"] / "validation.csv",  # TODO: your cached validation CSV
    "test": PATHS["splits"] / "test.csv",              # TODO: your cached test CSV
}

# Existing frozen PLM embeddings are loaded separately from the earlier .pt exports.
# TODO: fill in your own .pt cache paths if the filenames/folders below differ.
# These CSVs are not embedding caches; changing a filename does not convert a file format.
PLM_CACHE_FILES = {
    "ProtT5": PATHS["embeddings"] / "prot_t5_embedding_cache.pt",
    "ESM-2 650M": PATHS["embeddings"] / "esm2_t33_650M_snooppi_mean_pool_embeddings.pt",
    "ESM-C 600M": PATHS["embeddings"] / "esmc_600m_snooppi_mean_pool_embeddings.pt",
}
PATHS["split_files"] = SPLIT_CSV_FILES
PATHS["cache_files"] = PLM_CACHE_FILES
print("Data root:", PATHS["root"])
print("Split CSV files:", SPLIT_CSV_FILES)
print("PLM cache files:", PLM_CACHE_FILES)
# No directories are created until an explicit write switch is enabled.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from snooppi_protocol import (
    prepare_source_pairs, grouped_three_way_split, cold_target_split,
    interaction_components, load_reviewed_mapping, mapped_component_frame,
    mapped_overlap_report, summarize_protocol, write_protocol, matched_candidate_review,
)

WRITE_SPLITS = False
WRITE_NEGATIVE_CANDIDATES = False
SEED = 44
TARGET_SIDE = SEQUENCE_B  # Declare the biological target role before splitting.
SPLIT_FRACTIONS = {"train": .80, "validation": .10, "test": .10}
RUN_TAG = "seed44_v2"
PROTOCOLS_TO_PREVIEW = ["cold_target", "cold_protein"]
# Add "cold_cluster" / "cold_family" only after reviewing these metadata mappings.
CLUSTER_PATH = Path(os.environ.get("SNOOPPI_CLUSTER_FILE", str(PATHS["clusters"])))
ABLATION_SPLIT_ROOT = PATHS["root"] / "splits_ablations"
source_splits = load_splits(PATHS)
source_manifest = split_manifest(PATHS)
all_pairs, quarantined_duplicates = prepare_source_pairs(source_splits)
print("Duplicate/colliding pair rows quarantined (all copies):", len(quarantined_duplicates))
print("Unique pairs available:", len(all_pairs))
protocols = {}
protocol_configs = {}
protocol_reviews = {}
failures = {}


## 2. Grouping and feasibility

Shared helpers honor `SPLIT_FRACTIONS` as fractions of groups, not guaranteed
fractions of rows. The displayed row fractions and class counts must be reviewed
before export. Infeasible protocols are recorded and skipped independently; no
seed search based on test results is performed. All copies of duplicate/colliding
normalized pairs are quarantined with an exportable review table.


In [ ]:
def register_protocol(name, parts, group_column, extra=None, reviews=None):
    assert_model_ready(parts)
    protocols[name] = parts
    protocol_configs[name] = {"name": name, "seed": SEED, "requested_group_fractions": SPLIT_FRACTIONS,
                              "group_column": group_column, "target_side": TARGET_SIDE,
                              "leakage_identity": "canonical_X_spaces_removed_prefix1024_sha256_v1",
                              **(extra or {})}
    protocol_reviews[name] = {"quarantined_duplicates.csv": quarantined_duplicates, **(reviews or {})}
    display(summarize_protocol(name, parts, group_column))


## 3. Cold-target split

In [ ]:
if "cold_target" in PROTOCOLS_TO_PREVIEW:
    try:
        parts, rejected = cold_target_split(all_pairs, TARGET_SIDE, SEED, SPLIT_FRACTIONS)
        target_col = "protein_B_id" if TARGET_SIDE == SEQUENCE_B else "protein_A_id"
        register_protocol("cold_target", parts, target_col,
                          {"definition": "test targets absent from either partner role in train/validation; validation targets absent from train"},
                          {"cross_role_exclusions.csv": rejected})
        print("Cross-role rows excluded:", len(rejected))
    except ValueError as exc:
        failures["cold_target"] = str(exc)
        print("Cold-target preview unavailable:", exc)


## 4. Cold-protein connected-component split

If any protein participates in multiple pairs, assigning complete graph
components prevents either partner from crossing split boundaries. A giant
component can make the requested 80/10/10 proportions impossible; the
component-size table makes that limitation visible.

In [ ]:
if "cold_protein" in PROTOCOLS_TO_PREVIEW:
    component_frame = all_pairs.copy()
    component_frame["interaction_component"] = interaction_components(component_frame)
    component_sizes = component_frame.interaction_component.value_counts()
    display(component_sizes.describe().to_frame("pairs_per_component"))
    print("Largest component row fraction:", component_sizes.iloc[0] / len(component_frame))
    try:
        parts = grouped_three_way_split(component_frame, "interaction_component", SEED, SPLIT_FRACTIONS)
        register_protocol("cold_protein", parts, "interaction_component",
                          {"definition": "both partners isolated by normalized-protein interaction components"})
    except ValueError as exc:
        failures["cold_protein"] = str(exc)
        print("Cold-protein preview unavailable:", exc)


## 5. Reviewed cluster/family splits and metadata contracts

Cluster and family splitting links the groups of **both partners** into connected
components. This prevents a homolog/family appearing in another partner role
across partitions. Missing/ambiguous mappings fail, and giant components may
make the design infeasible. MMseqs2 assignments must be reviewed and mapped from
sequence IDs to full sequence hashes; a raw two-column TSV is not guessed.
Cross-species, temporal, and mutation designs remain contracts below.


In [ ]:
protein_metadata = pd.read_csv(PATHS["metadata"]) if PATHS["metadata"].is_file() else None
for protocol, path, field in [("cold_cluster", CLUSTER_PATH, "cluster_id"),
                              ("cold_family", PATHS["metadata"], "family")]:
    if protocol not in PROTOCOLS_TO_PREVIEW:
        continue
    try:
        mapping = load_reviewed_mapping(path, field, all_pairs)
        mapped_frame = mapped_component_frame(all_pairs, mapping, field)
        parts = grouped_three_way_split(mapped_frame, "mapped_component", SEED, SPLIT_FRACTIONS)
        overlap = mapped_overlap_report(parts, mapping, field)
        if overlap.filter(like="overlapping_").to_numpy().any():
            raise ValueError("Mapped groups overlap across partitions")
        register_protocol(protocol, parts, "mapped_component",
                          {"mapping_file": str(path), "mapping_sha256": file_sha256(path),
                           "mapping_field": field, "identity_scheme": "full_uppercase_spaces_removed_sha256_v1",
                           "definition": "connected components of both-partner reviewed groups"})
        display(overlap)
    except (ValueError, FileNotFoundError) as exc:
        failures[protocol] = str(exc)
        print(protocol, "preview unavailable:", exc)
print("Unavailable protocols:", failures)


## 6. Matched unknown-candidate review

Candidates match **selected training positives only**, in either partner
orientation. Labeled pairs are excluded against **all original partitions**;
missing biological metadata cannot count as a match. The output has no binary
label, keeps candidate provenance, and records that negative evidence is absent.
This is a matching queue, not a confirmed negative dataset or a sampling ratio.


In [ ]:
candidate_review = None
if PATHS["candidate_negatives"].is_file() and protein_metadata is not None:
    candidates = pd.read_csv(PATHS["candidate_negatives"])
    candidate_review, candidate_summary = matched_candidate_review(
        candidates, protein_metadata, source_splits["train"],
        pd.concat(source_splits.values(), ignore_index=True),
        f"{PATHS['candidate_negatives'].name}:sha256={file_sha256(PATHS['candidate_negatives'])}",
    )
    candidate_review["training_split_sha256"] = source_manifest["sha256"]["train"]
    candidate_review["metadata_sha256"] = file_sha256(PATHS["metadata"])
    display(candidate_review.drop(columns=[SEQUENCE_A, SEQUENCE_B]).head(20))
    print(candidate_summary)
else:
    print("Matching skipped; reviewed candidate and metadata files are required.")


## 7. Optional, non-overwriting exports

In [ ]:
if WRITE_SPLITS:
    unavailable = set(PROTOCOLS_TO_PREVIEW) - set(protocols)
    if unavailable:
        raise RuntimeError(f"Requested protocols are not exportable: {unavailable}; review failures or select feasible protocols")
    # Preflight every destination before writing any protocol.
    destinations = {name: ABLATION_SPLIT_ROOT / f"{name}_{RUN_TAG}" for name in protocols}
    from snooppi_utils import check_output_paths
    check_output_paths(destinations.values())
    for name, parts in protocols.items():
        write_protocol(destinations[name], parts, source_manifest,
                       protocol_configs[name], protocol_reviews[name])
        print("Exported reviewed protocol:", destinations[name])
else:
    print("Preview only. No split CSV or manifest was written.")

if WRITE_NEGATIVE_CANDIDATES:
    if candidate_review is None:
        raise RuntimeError("No candidate review table was built")
    output_path = ABLATION_SPLIT_ROOT / f"matched_candidates_{RUN_TAG}.csv"
    manifest_path = output_path.with_suffix(".manifest.json")
    prepare_outputs([output_path, manifest_path])
    candidate_review.to_csv(output_path, index=False)
    save_json({"source_splits": source_manifest, "metadata_sha256": file_sha256(PATHS["metadata"]),
               "candidate_input_sha256": file_sha256(PATHS["candidate_negatives"]),
               "candidate_summary": candidate_summary,
               "status": "unknown_requires_review; no confirmed negatives"}, manifest_path)
    print("Saved unknown review queue:", output_path)


## Protocols that still require study-specific implementation

- **Cross-species:** declare source/test species and handle homologs and mixed-species pairs explicitly.
- **Temporal:** use pair-level assay/evidence dates and a fixed cutoff; a protein collection date does not establish when an interaction became known.
- **Mutation:** keep wild-type/mutant relatives together and evaluate held-out variants with a frozen checkpoint.

These designs are not produced by this notebook. Exported cold-target,
cold-protein, cluster and family splits preserve `original_split` and label
evidence, plus quarantine/exclusion tables. Repartitioning original test pairs
creates a new benchmark; historical test scores are not directly comparable.
After review, select one exported directory with `SNOOPPI_SPLIT_DIR` and rerun
00 before 01/03/05. Cached frozen embeddings can be reused if coverage passes.
